# E21_synthetic_pairs — training

> 🔴 Target side must ALWAYS be genuine organizers' text. Augment TRAIN ONLY.

**Read `EXPERIMENT.md` first** — it has what this must beat and how to read the outcome.
Record results in **this folder's `RESULTS.md`**, not only the top-level one.

## 🔴 Non-negotiables
| | |
|---|---|
| **Keep EVERY arm's `best/`** | including the losers — a usefully-*disagreeing* model is what E14/E19 need |
| **bf16 on sm_80+, else fp32** | 🔴 **never fp16** — T5 goes NaN *silently* and still writes a valid CSV |
| **`--seed 42 --dev-size 5000`** | the frozen split. Change it and nothing is comparable |
| **Select on composite, not loss** | one run's lowest loss coincided with its *worst* Token F1 |
| **`run.json` beside the weights** | `checkpoint_hash` is not a run identity |


In [ ]:
# 1 ── CONFIG — the only cell you should need to edit
EXPERIMENT = "E21_synthetic_pairs"
DATA       = "../data/draft_only"        # switch to the Tier-1 winner where EXPERIMENT.md says so
MAX_SRC, MAX_TGT = 384, 256
STEPS      = 4000
EVAL_EVERY = 250
BATCH, ACCUM = 8, 8                  # effective 64 — raise BATCH / lower ACCUM on a big GPU

# (arm_dir, model, extra flags). 🔴 EVERY arm's checkpoint is kept.
ARMS = [
    ("A0_gold_control", "csebuetnlp/banglat5", ""),
    ("A1_backtranslated", "csebuetnlp/banglat5", ""),
    ("A2_tagged", "csebuetnlp/banglat5", ""),
    ("B_pseudolabel", "csebuetnlp/banglat5", ""),
    ("C_teacher_targets", "csebuetnlp/banglat5", ""),
]

import os, json, subprocess, sys, time, shutil
from pathlib import Path
print(f'{EXPERIMENT}: {len(ARMS)} arm(s)')
for a, m, f in ARMS: print(f'  {a:24s} {m:38s} {f}')


In [ ]:
# 2 ── environment gate
import torch
assert torch.cuda.is_available(), 'no GPU'
cap = torch.cuda.get_device_capability()
BF16 = cap[0] >= 8      # 🔴 gate on capability, NOT is_bf16_supported() — that returns True on a T4 (emulated, slower than fp32)
PRECISION = 'bf16' if BF16 else 'fp32'
print(f'{torch.cuda.device_count()}x {torch.cuda.get_device_name(0)} | sm_{cap[0]}{cap[1]} | precision {PRECISION}')

import transformers
print('transformers', transformers.__version__)
# 🔴 4.57.3 for BanglaT5/T5. Newer is required for Qwen3/Gemma — see the note in cell 4.
assert Path(DATA, 'train.parquet').is_file(), f'no train.parquet in {DATA}'
import pandas as pd
for s in ('train','dev','test'):
    print(f'  {s}: {len(pd.read_parquet(Path(DATA,s+".parquet"))):,} rows')


In [ ]:
# 3 ── train every arm. Each gets its OWN directory with best/ + run.json.
# 🔴 Do NOT trade quality for throughput. If memory is tight: more GPUs, smaller BATCH with
#    higher ACCUM, or gradient checkpointing. NEVER fp16, never a shorter run, never LoRA
#    in place of a full fine-tune, never a shorter sequence length.
for arm, model, extra in ARMS:
    out = Path(arm); out.mkdir(exist_ok=True)
    if (out/'best'/'config.json').is_file():
        print(f'== {arm}: best/ exists, skipping (delete to rerun)'); continue
    cmd = [sys.executable, '../code/02_train_t5.py',
           '--data-dir', DATA, '--model', model, '--out', str(out),
           '--max-source-len', str(MAX_SRC), '--max-target-len', str(MAX_TGT),
           '--batch-size', str(BATCH), '--grad-accum', str(ACCUM),
           '--eval-steps', str(EVAL_EVERY), '--eval-subset', '300',
           '--precision', PRECISION, '--no-group-by-length',
           '--gen-num-beams', '4', '--gen-min-new-tokens', '80',
           '--max-steps', str(STEPS)] + ([] if not extra else extra.split())
    print(f'\n== {arm}\n   ' + ' '.join(cmd), flush=True)
    t0 = time.time(); subprocess.run(cmd, check=True)
    print(f'   {arm} done in {(time.time()-t0)/60:.1f} min')


## Notes

Everything here is encoder-decoder, so `02_train_t5.py` handles every arm.


In [ ]:
# 4 ── score every arm on the frozen dev split and write its record
for arm, model, extra in ARMS:
    out = Path(arm)
    if not (out/'best'/'config.json').is_file():
        print(f'{arm}: no best/ — skipped'); continue
    subprocess.run([sys.executable, '../code/04_decode.py',
        '--ckpt', str(out/'best'), '--data-dir', DATA, '--split', 'dev',
        '--limit', '300', '--mode', 'beam', '--num-beams', '4',
        '--min-new-tokens', '80', '--max-new-tokens', '320',
        '--length-penalty', '1.0', '--no-bertscore',
        '--record', str(out/'dev.json')], check=True)
    print(f'{arm}: scored -> {out}/dev.json')


In [ ]:
# 5 ── summary to paste into THIS FOLDER's RESULTS.md
import pandas as pd
rows = []
for arm, model, extra in ARMS:
    d = Path(arm)/'dev.json'
    if not d.is_file(): rows.append(dict(arm=arm, model=model, note='no result')); continue
    r = json.load(open(d, encoding='utf-8'))
    ckpt = (Path(arm)/'best').resolve()
    rows.append(dict(arm=arm, model=model,
        token_f1=round(r.get('token_f1', float('nan')), 4),
        rouge_l=round(r.get('rouge_l', float('nan')), 4),
        tokens=round(r.get('pred_tokens', float('nan')), 1),
        ckpt_kept=(Path(arm)/'best'/'config.json').is_file(), ckpt=str(ckpt)))
df = pd.DataFrame(rows).sort_values('token_f1', ascending=False, na_position='last')
print(df.to_markdown(index=False))
print('\nincumbent: Token F1 0.7724 | ROUGE-L 0.7324 | noise floor 0.0044')
print('LB ~ 0.4646 + 0.3098*TokenF1 + 0.2*ROUGE-L')
missing = [r['arm'] for r in rows if not r.get('ckpt_kept')]
assert not missing, f'🔴 checkpoint MISSING for {missing} — these cannot be submitted without a retrain'
print('\n✅ every arm has a checkpoint. Record all of this in ./RESULTS.md')
